In [ ]:
# AI-Assisted Concept Art Generator for Google Colab
# Updated for Gemini 2.5 API - April 2026

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: INSTALL REQUIRED PACKAGES
# ═══════════════════════════════════════════════════════════════════════════════

print("📦 Installing required packages...")
print("=" * 70)

!pip install -q google-generativeai gradio pillow

print("✅ All packages installed successfully!\n")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: IMPORT LIBRARIES
# ═══════════════════════════════════════════════════════════════════════════════

print("📚 Importing libraries...")
print("=" * 70)

import google.generativeai as genai
import gradio as gr
import json
import re
import random
import math
from datetime import datetime
from typing import List, Dict, Tuple, Optional
import io
from PIL import Image, ImageDraw, ImageFont
import base64

print("✅ Libraries imported successfully!\n")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 3: MODEL CONFIGURATION (UPDATED FOR GEMINI 2.5 - APRIL 2026)
# ═══════════════════════════════════════════════════════════════════════════════

# Current available models (April 2026) - UPDATED
AVAILABLE_MODELS = [
    "gemini-2.5-flash",          # ✅ PRIMARY - Latest stable model (April 2026)
    "gemini-2.5-pro",            # Fallback - More powerful for complex scenes
    "gemini-2.0-flash",          # Alternative - Stable production model
]

# Primary model to use - UPDATED to gemini-2.5-flash
PRIMARY_MODEL = "gemini-2.5-flash"

print(f"🤖 Configured to use: {PRIMARY_MODEL}")
print("=" * 70)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: SCENE GENERATION LOGIC (UPDATED API SYNTAX)
# ═══════════════════════════════════════════════════════════════════════════════

class SceneGenerator:
    """Handles scene generation using Gemini 2.5 API with latest error handling."""

    def __init__(self, model_name: str = PRIMARY_MODEL):
        self.model_name = model_name
        self.model = None
        self._initialize_model()

    def _initialize_model(self):
        """Initialize the generative model with intelligent fallback."""
        try:
            self.model = genai.GenerativeModel(self.model_name)
            print(f"✅ Model initialized: {self.model_name}")
        except Exception as e:
            print(f"⚠️  Warning: Could not initialize {self.model_name}")
            print(f"Error: {str(e)}")
            # Try fallback models automatically
            for fallback_model in AVAILABLE_MODELS[1:]:
                try:
                    print(f"🔄 Trying fallback model: {fallback_model}")
                    self.model = genai.GenerativeModel(fallback_model)
                    self.model_name = fallback_model
                    print(f"✅ Successfully using fallback: {fallback_model}")
                    break
                except Exception as e2:
                    print(f"❌ {fallback_model} also failed: {str(e2)}")
                    continue

    def list_available_models(self) -> List[str]:
        """List all available models for debugging."""
        try:
            models = genai.list_models()
            available = [m.name for m in models if 'generateContent' in m.supported_generation_methods]
            return available
        except Exception as e:
            print(f"Could not list models: {str(e)}")
            return []

    def expand_prompt_to_scenes(self, prompt: str, style: str) -> List[Dict]:
        """
        Expand a user prompt into 4 detailed scene descriptions.
        Updated for Gemini 2.5 API with enhanced generation config.

        Args:
            prompt: User's creative concept
            style: Art style preference

        Returns:
            List of scene dictionaries with title, description, mood, and elements
        """
        if not self.model:
            raise Exception("Model not initialized. Please check your API key and internet connection.")

        try:
            system_prompt = f"""You are an expert concept art director with deep knowledge of visual storytelling and scene composition.

Given this creative prompt: "{prompt}"
Art Style: {style}

Your task is to create exactly 4 diverse, visually stunning scene descriptions that could be turned into concept art. Each scene should:

1. Be VISUALLY DISTINCT - showcase different aspects, angles, or moments
2. Include SPECIFIC DETAILS about environment, lighting, atmosphere, and key elements
3. Be SUITABLE FOR IMAGE GENERATION - describe what would actually be visible
4. Work together to tell a COHESIVE STORY
5. Match the {style} aesthetic

Format your response as valid JSON (no markdown, no code blocks, just raw JSON):
{{
  "scenes": [
    {{
      "title": "Compelling 3-5 word scene title",
      "description": "Detailed 2-3 sentence visual description with specific details about lighting, composition, colors, and atmosphere",
      "mood": "Single word emotional tone (e.g., ominous, serene, chaotic, mystical)",
      "keyElements": ["specific element 1", "specific element 2", "specific element 3", "specific element 4"]
    }}
  ]
}}

Make each scene cinematically interesting and visually rich. Think like a film director choosing the perfect shots."""

            # UPDATED: Gemini 2.5 API syntax with enhanced generation config
            response = self.model.generate_content(
                system_prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.9,
                    top_p=0.95,
                    top_k=40,
                    max_output_tokens=2048,
                    # New for Gemini 2.5: response_mime_type for structured output
                    response_mime_type="application/json",
                ),
                # New for Gemini 2.5: Safety settings update
                safety_settings=[
                    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_MEDIUM_AND_ABOVE"},
                    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_MEDIUM_AND_ABOVE"},
                    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_MEDIUM_AND_ABOVE"},
                    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_MEDIUM_AND_ABOVE"},
                ]
            )

            response_text = response.text.strip()

            # Clean up the response - remove markdown code blocks if present
            response_text = re.sub(r'```json\s*|\s*```', '', response_text)
            response_text = response_text.strip()

            # Parse JSON
            data = json.loads(response_text)
            scenes = data.get('scenes', [])

            # Validate we have exactly 4 scenes
            if len(scenes) < 4:
                raise ValueError(f"Expected 4 scenes, got {len(scenes)}. Please try again.")

            return scenes[:4]  # Return exactly 4 scenes

        except json.JSONDecodeError as e:
            print(f"JSON Parse Error: {e}")
            print(f"Response text: {response_text[:500]}")
            raise Exception("Failed to parse scene data. The AI response was not valid JSON. Please try again.")
        except Exception as e:
            error_msg = str(e)
            if "404" in error_msg or "not found" in error_msg.lower():
                # Model not found error
                available = self.list_available_models()
                if available:
                    raise Exception(f"Model '{self.model_name}' not found. Available models: {', '.join(available[:5])}")
                else:
                    raise Exception(f"Model '{self.model_name}' not found. Please check your API key and internet connection.")
            elif "API key" in error_msg or "authentication" in error_msg.lower():
                raise Exception("Invalid API key. Please check your Google API key and try again.")
            else:
                raise Exception(f"Failed to generate scenes: {str(e)}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: ABSTRACT ART GENERATION
# ═══════════════════════════════════════════════════════════════════════════════

class AbstractArtGenerator:
    """Generates abstract concept art using PIL."""

    # Color palettes based on moods
    COLOR_PALETTES = {
        'dark': ['#1a1a2e', '#16213e', '#0f3460', '#533483', '#2d1b4e'],
        'ominous': ['#1a0000', '#330000', '#4d0000', '#660000', '#1a1a1a'],
        'bright': ['#f39c12', '#e74c3c', '#3498db', '#2ecc71', '#f1c40f'],
        'cheerful': ['#ff6b6b', '#4ecdc4', '#45b7d1', '#ffa07a', '#98d8c8'],
        'mystical': ['#9b59b6', '#8e44ad', '#663399', '#4a148c', '#6a1b9a'],
        'magical': ['#e91e63', '#9c27b0', '#673ab7', '#3f51b5', '#536dfe'],
        'serene': ['#00bcd4', '#0097a7', '#00796b', '#00897b', '#26a69a'],
        'calm': ['#607d8b', '#546e7a', '#455a64', '#37474f', '#78909c'],
        'chaotic': ['#ff5722', '#ff6f00', '#f44336', '#d32f2f', '#c62828'],
        'intense': ['#d50000', '#ff1744', '#f50057', '#c51162', '#aa00ff'],
        'warm': ['#ff6b6b', '#ee5a6f', '#c44569', '#f8b500', '#ff8c42'],
        'cozy': ['#d4a574', '#a67c52', '#8b6f47', '#f4e4c1', '#e8c4a0'],
        'cold': ['#2c3e50', '#34495e', '#1abc9c', '#3498db', '#5dade2'],
        'ethereal': ['#e1bee7', '#ce93d8', '#ba68c8', '#ab47bc', '#d1c4e9'],
        'vibrant': ['#ff6b6b', '#feca57', '#48dbfb', '#ff9ff3', '#54a0ff'],
        'mysterious': ['#2d132c', '#433d3c', '#554348', '#6e5773', '#801336'],
        'peaceful': ['#a8e6cf', '#dcedc1', '#ffd3b6', '#ffaaa5', '#b8e0d2'],
        'dramatic': ['#c0392b', '#8e44ad', '#2c3e50', '#d35400', '#7f8c8d'],
        'default': ['#e67e22', '#16a085', '#2980b9', '#8e44ad', '#27ae60']
    }

    def __init__(self, width: int = 800, height: int = 600):
        self.width = width
        self.height = height

    def detect_mood_palette(self, mood: str) -> List[str]:
        """Detect the color palette based on mood keyword."""
        mood_lower = mood.lower()

        for palette_name, colors in self.COLOR_PALETTES.items():
            if palette_name in mood_lower or mood_lower in palette_name:
                return colors

        return self.COLOR_PALETTES['default']

    def seeded_random(self, seed: int, min_val: float, max_val: float, offset: int = 0) -> float:
        """Generate deterministic random value using seed."""
        x = math.sin(seed * 12.9898 + offset * 78.233) * 43758.5453
        return min_val + (x - math.floor(x)) * (max_val - min_val)

    def generate_abstract_art(self, scene_id: int, mood: str, title: str) -> Image.Image:
        """
        Generate abstract concept art for a scene.

        Args:
            scene_id: Unique identifier for deterministic generation
            mood: Scene mood for color selection
            title: Scene title for annotation

        Returns:
            PIL Image object
        """
        # Create image with gradient background
        img = Image.new('RGB', (self.width, self.height), color='#0a0a0f')
        draw = ImageDraw.Draw(img, 'RGBA')

        # Get color palette
        palette = self.detect_mood_palette(mood)

        # Generate gradient background
        for y in range(self.height):
            ratio = y / self.height
            color1 = self._hex_to_rgb(palette[0])
            color2 = self._hex_to_rgb(palette[2])
            color = tuple(int(color1[i] + (color2[i] - color1[i]) * ratio) for i in range(3))
            draw.line([(0, y), (self.width, y)], fill=color)

        # Generate abstract shapes
        num_shapes = int(self.seeded_random(scene_id, 15, 25, 0))

        for i in range(num_shapes):
            shape_type = int(self.seeded_random(scene_id, 0, 4, i * 3))
            x = self.seeded_random(scene_id, 0, self.width, i * 2)
            y = self.seeded_random(scene_id, 0, self.height, i * 2 + 1)
            size = self.seeded_random(scene_id, 30, 150, i * 4)
            color_idx = int(self.seeded_random(scene_id, 0, len(palette), i * 5))
            color = self._hex_to_rgb(palette[color_idx])
            opacity = int(self.seeded_random(scene_id, 50, 180, i * 6))

            if shape_type == 0:  # Circle
                draw.ellipse(
                    [x - size/2, y - size/2, x + size/2, y + size/2],
                    fill=color + (opacity,)
                )
            elif shape_type == 1:  # Rectangle
                draw.rectangle(
                    [x - size/2, y - size/2, x + size/2, y + size/2],
                    fill=color + (opacity,)
                )
            elif shape_type == 2:  # Triangle
                points = [
                    (x, y - size/2),
                    (x - size/2, y + size/2),
                    (x + size/2, y + size/2)
                ]
                draw.polygon(points, fill=color + (opacity,))
            else:  # Line
                x2 = self.seeded_random(scene_id, 0, self.width, i * 8)
                y2 = self.seeded_random(scene_id, 0, self.height, i * 9)
                width = int(self.seeded_random(scene_id, 2, 8, i * 10))
                draw.line([(x, y), (x2, y2)], fill=color + (opacity,), width=width)

        # Add overlay texture
        overlay = Image.new('RGBA', (self.width, self.height), (0, 0, 0, 0))
        overlay_draw = ImageDraw.Draw(overlay)

        for _ in range(500):
            px = self.seeded_random(scene_id, 0, self.width, random.randint(0, 10000))
            py = self.seeded_random(scene_id, 0, self.height, random.randint(0, 10000))
            overlay_draw.point((px, py), fill=(255, 255, 255, 30))

        img = Image.alpha_composite(img.convert('RGBA'), overlay).convert('RGB')

        # Add title text
        draw = ImageDraw.Draw(img)
        try:
            # Try to use a nice font if available, otherwise use default
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 36)
        except:
            font = ImageFont.load_default()

        # Add semi-transparent background for text
        text_bbox = draw.textbbox((0, 0), title, font=font)
        text_width = text_bbox[2] - text_bbox[0]
        text_height = text_bbox[3] - text_bbox[1]

        padding = 20
        text_bg = [
            (self.width - text_width) // 2 - padding,
            self.height - text_height - 40 - padding,
            (self.width + text_width) // 2 + padding,
            self.height - 40 + padding
        ]
        draw.rectangle(text_bg, fill=(0, 0, 0, 180))

        # Draw title
        text_position = ((self.width - text_width) // 2, self.height - text_height - 40)
        draw.text(text_position, title, fill='#ffffff', font=font)

        return img

    def _hex_to_rgb(self, hex_color: str) -> Tuple[int, int, int]:
        """Convert hex color to RGB tuple."""
        hex_color = hex_color.lstrip('#')
        return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: STORYBOARD GENERATOR
# ═══════════════════════════════════════════════════════════════════════════════

class StoryboardGenerator:
    """Generates complete storyboard with all scenes."""

    def __init__(self):
        self.art_generator = AbstractArtGenerator(width=1200, height=800)

    def create_storyboard(self, scenes: List[Dict]) -> Tuple[List[Image.Image], str]:
        """
        Create storyboard images and text summary.

        Args:
            scenes: List of scene dictionaries

        Returns:
            Tuple of (list of images, text summary)
        """
        images = []
        text_parts = []

        text_parts.append("=" * 80)
        text_parts.append("CONCEPT ART STORYBOARD")
        text_parts.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        text_parts.append("=" * 80)
        text_parts.append("")

        for i, scene in enumerate(scenes, 1):
            # Generate artwork
            img = self.art_generator.generate_abstract_art(
                scene_id=i * 100,
                mood=scene['mood'],
                title=f"Scene {i}: {scene['title']}"
            )
            images.append(img)

            # Generate text description
            text_parts.append(f"\n{'=' * 80}")
            text_parts.append(f"SCENE {i}: {scene['title'].upper()}")
            text_parts.append(f"{'=' * 80}")
            text_parts.append(f"\nMood: {scene['mood'].capitalize()}")
            text_parts.append(f"\nDescription:")
            text_parts.append(scene['description'])
            text_parts.append(f"\nKey Elements:")
            for element in scene.get('keyElements', []):
                text_parts.append(f"  • {element}")
            text_parts.append("")

        text_parts.append("=" * 80)
        text_parts.append("END OF STORYBOARD")
        text_parts.append("=" * 80)

        return images, "\n".join(text_parts)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7: GRADIO INTERFACE (UPDATED FOR GEMINI 2.5)
# ═══════════════════════════════════════════════════════════════════════════════

class ConceptArtApp:
    """Main application with Gradio interface - Updated April 2026."""

    def __init__(self):
        self.scene_generator = None
        self.storyboard_generator = StoryboardGenerator()
        self.current_scenes = []
        self.api_configured = False

        # Style descriptions
        self.style_presets = {
            "Cinematic": "Dramatic lighting, film-quality aesthetics with deep shadows and highlights",
            "Painterly": "Artistic brushstrokes, traditional concept art style with visible texture",
            "Sci-Fi": "Futuristic, cyberpunk atmosphere with neon lights and technology",
            "Fantasy": "Magical, ethereal environment with mystical elements and wonder",
            "Realistic": "Photorealistic, natural lighting with attention to detail"
        }

    def configure_api(self, api_key: str) -> str:
        """Configure the API with user's key - Updated for Gemini 2.5."""
        if not api_key or api_key.strip() == "":
            return "❌ Please enter a valid API key"

        try:
            genai.configure(api_key=api_key.strip())
            self.scene_generator = SceneGenerator(model_name=PRIMARY_MODEL)

            # UPDATED: Test the API with Gemini 2.5 compatible request
            test_model = genai.GenerativeModel(PRIMARY_MODEL)
            test_response = test_model.generate_content(
                "Hello",
                generation_config=genai.types.GenerationConfig(
                    max_output_tokens=50,
                    temperature=0.1
                )
            )

            if test_response and test_response.text:
                self.api_configured = True
                return f"✅ API configured successfully!\n✅ Using model: {PRIMARY_MODEL} (Gemini 2.5)\n\nYou can now generate concept art."
            else:
                return "❌ API key accepted but model test failed. Please try again."

        except Exception as e:
            self.api_configured = False
            error_msg = str(e)

            if "404" in error_msg or "not found" in error_msg:
                # Try to list available models
                try:
                    models = genai.list_models()
                    available = [m.name for m in models if 'generateContent' in m.supported_generation_methods]
                    if available:
                        model_list = "\n".join([f"  - {m}" for m in available[:5]])
                        return f"❌ Model '{PRIMARY_MODEL}' not found.\n\nAvailable models:\n{model_list}\n\nPlease check your API access or try a different model."
                except:
                    pass
                return f"❌ Model '{PRIMARY_MODEL}' not found. The model may not be available in your region.\n\nPlease check the Google AI documentation for current models."

            elif "API" in error_msg or "key" in error_msg.lower():
                return f"❌ Invalid API key.\n\nError: {error_msg}\n\nPlease check your API key at:\nhttps://makersuite.google.com/app/apikey"

            else:
                return f"❌ Error: {error_msg}\n\nPlease check:\n1. Your API key is correct\n2. You have internet connection\n3. The Gemini API is accessible"

    def generate_scenes(self, prompt: str, style: str, progress=gr.Progress()) -> Tuple[str, str]:
        """Generate scenes from user prompt using Gemini 2.5."""
        if not self.api_configured:
            return "❌ Please configure your API key first!", ""

        if not prompt or prompt.strip() == "":
            return "❌ Please enter a creative concept or prompt!", ""

        try:
            progress(0, desc="🎨 Analyzing your concept...")

            # Generate scenes with Gemini 2.5
            self.current_scenes = self.scene_generator.expand_prompt_to_scenes(
                prompt=prompt.strip(),
                style=style
            )

            progress(0.5, desc="📝 Formatting scenes...")

            # Format scenes for display
            scene_text = self._format_scenes_display(self.current_scenes)

            progress(1.0, desc="✅ Complete!")

            return (
                "✅ Successfully generated 4 unique scenes!",
                scene_text
            )

        except Exception as e:
            return f"❌ Error: {str(e)}", ""

    def generate_artwork(self, progress=gr.Progress()) -> Tuple[List[Image.Image], str, str]:
        """Generate artwork for all scenes."""
        if not self.current_scenes:
            return [], "❌ Please generate scenes first!", ""

        try:
            progress(0, desc="🎨 Creating concept artwork...")

            images, storyboard_text = self.storyboard_generator.create_storyboard(
                self.current_scenes
            )

            progress(1.0, desc="✅ Artwork complete!")

            return (
                images,
                "✅ Generated 4 concept artworks successfully!",
                storyboard_text
            )

        except Exception as e:
            return [], f"❌ Error: {str(e)}", ""

    def _format_scenes_display(self, scenes: List[Dict]) -> str:
        """Format scenes for nice display."""
        output = []

        for i, scene in enumerate(scenes, 1):
            output.append(f"### 🎬 Scene {i}: {scene['title']}")
            output.append(f"**Mood:** {scene['mood'].capitalize()}")
            output.append(f"\n{scene['description']}")
            output.append(f"\n**Key Elements:**")
            for element in scene.get('keyElements', []):
                output.append(f"- {element}")
            output.append("\n---\n")

        return "\n".join(output)

    def create_interface(self):
        """Create the Gradio interface - Updated April 2026."""

        # Custom CSS for beautiful styling
        custom_css = """
        .gradio-container {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
        }
        .header-text {
            text-align: center;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-clip: text;
            font-size: 2.5em;
            font-weight: bold;
            margin-bottom: 10px;
        }
        .subtitle-text {
            text-align: center;
            color: #666;
            font-size: 1.2em;
            margin-bottom: 30px;
        }
        """

        with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="AI Concept Art Generator") as interface:

            # Header
            gr.Markdown(
                f"""
                <div class="header-text">🎨 AI-Assisted Concept Art Generator</div>
                <div class="subtitle-text">Transform your creative vision into stunning visual storyboards</div>
                <div style="text-align: center; color: #999; font-size: 0.9em; margin-bottom: 20px;">
                Powered by Google Gemini 2.5 | Updated April 2026
                </div>
                """
            )

            # API Configuration Section
            with gr.Accordion("🔑 Step 1: API Configuration", open=True):
                gr.Markdown(
                    """
                    **Get your free Google API key:**
                    1. Visit [Google AI Studio](https://makersuite.google.com/app/apikey)
                    2. Click "Create API Key"
                    3. Copy and paste it below

                    **Note:** Now using Gemini 2.5 Flash for enhanced performance!
                    """
                )

                with gr.Row():
                    api_key_input = gr.Textbox(
                        label="Google API Key",
                        placeholder="Enter your API key here (starts with AIza...)...",
                        type="password",
                        scale=3
                    )
                    config_btn = gr.Button("Configure API", variant="primary", scale=1)

                api_status = gr.Textbox(label="Status", interactive=False, lines=5)

                config_btn.click(
                    fn=self.configure_api,
                    inputs=[api_key_input],
                    outputs=[api_status]
                )

            # Scene Generation Section
            with gr.Accordion("✨ Step 2: Generate Scene Descriptions", open=True):
                gr.Markdown(
                    """
                    Describe your creative concept. Be specific about setting, mood, and atmosphere.

                    **Example prompts:**
                    - "A post-apocalyptic city street at dusk with neon signs flickering"
                    - "An ancient magical library floating among the clouds"
                    - "Underwater research facility with bioluminescent creatures"
                    """
                )

                prompt_input = gr.Textbox(
                    label="Your Creative Concept",
                    placeholder="Describe your vision...",
                    lines=3
                )

                style_selector = gr.Radio(
                    choices=list(self.style_presets.keys()),
                    label="Art Style",
                    value="Cinematic",
                    info="Choose the aesthetic direction for your concept art"
                )

                generate_scenes_btn = gr.Button(
                    "🎬 Generate Scene Breakdown",
                    variant="primary",
                    size="lg"
                )

                scene_status = gr.Textbox(label="Generation Status", interactive=False)
                scenes_output = gr.Markdown(label="Generated Scenes")

                generate_scenes_btn.click(
                    fn=self.generate_scenes,
                    inputs=[prompt_input, style_selector],
                    outputs=[scene_status, scenes_output]
                )

            # Artwork Generation Section
            with gr.Accordion("🖼️ Step 3: Generate Concept Artwork", open=True):
                gr.Markdown(
                    """
                    Transform your scene descriptions into visual concept art.
                    This will create 4 unique artworks based on the generated scenes.
                    """
                )

                generate_art_btn = gr.Button(
                    "🎨 Create Concept Artwork",
                    variant="primary",
                    size="lg"
                )

                art_status = gr.Textbox(label="Generation Status", interactive=False)

                artwork_gallery = gr.Gallery(
                    label="Generated Concept Art",
                    columns=2,
                    rows=2,
                    height="auto",
                    object_fit="contain"
                )

                generate_art_btn.click(
                    fn=self.generate_artwork,
                    inputs=[],
                    outputs=[artwork_gallery, art_status, gr.Textbox(visible=False)]
                )

            # Storyboard Export Section
            with gr.Accordion("📥 Step 4: Export Storyboard", open=False):
                gr.Markdown(
                    """
                    Download your complete storyboard including all scene descriptions and artwork details.
                    """
                )

                storyboard_text = gr.Textbox(
                    label="Storyboard Text",
                    lines=20,
                    interactive=False
                )

                # Connect storyboard generation to text output
                generate_art_btn.click(
                    fn=self.generate_artwork,
                    inputs=[],
                    outputs=[gr.Gallery(visible=False), gr.Textbox(visible=False), storyboard_text]
                )

            # Footer
            gr.Markdown(
                f"""
                ---
                ### 💡 Tips for Best Results:
                - **Be specific**: Include details about lighting, atmosphere, and key visual elements
                - **Set the scene**: Describe time of day, weather, and environmental conditions
                - **Think visually**: Focus on what would actually be visible in the scene
                - **Iterate**: Don't hesitate to regenerate if the first results aren't perfect

                ### 🎯 Current Configuration:
                - **Model**: {PRIMARY_MODEL} (Gemini 2.5 Flash)
                - **API Version**: v1 (stable)
                - **Last Updated**: April 2026

                **Built with ❤️ using Google Gemini 2.5 & Gradio**
                """
            )

        return interface

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 8: LAUNCH APPLICATION
# ═══════════════════════════════════════════════════════════════════════════════

print("\n" + "=" * 70)
print("🚀 LAUNCHING CONCEPT ART GENERATOR")
print("=" * 70)
print(f"📱 Using Model: {PRIMARY_MODEL} (Gemini 2.5)")
print("=" * 70)

# Create and launch the app
app = ConceptArtApp()
interface = app.create_interface()

print("\n📱 Starting Gradio interface...")
print("=" * 70)

# Launch with share=True to get a public URL
interface.launch(
    share=True,
    debug=True,
    show_error=True
)

print("\n✅ Application is now running!")
print("=" * 70)

📦 Installing required packages...
✅ All packages installed successfully!

📚 Importing libraries...
✅ Libraries imported successfully!

🤖 Configured to use: gemini-2.5-flash

🚀 LAUNCHING CONCEPT ART GENERATOR
📱 Using Model: gemini-2.5-flash (Gemini 2.5)


/tmp/ipykernel_1696/1083571950.py:568: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="AI Concept Art Generator") as interface:
/tmp/ipykernel_1696/1083571950.py:568: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title="AI Concept Art Generator") as interface:



📱 Starting Gradio interface...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b911e1ee57a95cd5d8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


✅ Model initialized: gemini-2.5-flash


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3145.23ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2263.93ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1308.52ms


...

JSON Parse Error: Unterminated string starting at: line 17 column 22 (char 1831)
Response text: {
  "scenes": [
    {
      "title": "Cathedral's Heart Revealed",
      "description": "A breathtaking wide shot from a high vantage point, overlooking the main chamber of the forgotten library. Golden shafts of sunlight, filtered through a massive, intricately patterned amber and emerald stained-glass rose window high above, illuminate a central ancient oak tree whose enormous trunk forms the core of a winding spiral staircase. Towering shelves of rich, forest-green and amber leather-bound boo


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2843.70ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1210.08ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1261.05ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 12878.95ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 5006.43ms


✅ Model initialized: gemini-2.5-flash


✅ Model initialized: gemini-2.5-flash


✅ Model initialized: gemini-2.5-flash
